# 18. Deep Vision Classification of CWT Spectrograms
**Objective:** Pass unflattened 2D Time-Frequency spectrograms into a custom ResNet-style architecture to leverage true spatial/temporal translational invariance.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR
from src.cwt_features import extract_cwt_spectrograms
from src.cwt_net import train_cwt_vision_model

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load Data & Extract CWT Tensors

In [2]:
file_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1.h5")
with h5py.File(file_path, 'r') as f:
    X_bal = np.array(f['X'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

# Generating the 4D Tensor: (Windows, Scales, Time, Channels)
X_cwt = extract_cwt_spectrograms(X_bal, n_scales=16, n_jobs=-1)

X_train_cwt, y_train = X_cwt[train_idx], y_bal[train_idx]
X_test_cwt, y_test = X_cwt[test_idx], y_bal[test_idx]

print(f"CWT Train Shape: {X_train_cwt.shape}")
print(f"CWT Test Shape: {X_test_cwt.shape}")

Starting parallel CWT extraction on 18630 windows...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1833660279170309s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  62 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.06270170211791992s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1371901035308838s.) Setting batch_size=8.
[Parallel(n_jobs=-1)]: Done 128 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done 188 tasks      | elapsed:    1.4s
[Parallel(n_jobs=

CWT extraction complete. Spectrogram tensor shape: (18630, 16, 20, 10)
CWT Train Shape: (12664, 16, 20, 10)
CWT Test Shape: (5444, 16, 20, 10)


### 2. Train and Evaluate CWT Deep Vision Network

In [3]:
acc, f1, cwt_model = train_cwt_vision_model(
    X_train=X_train_cwt, 
    y_train=y_train, 
    X_test=X_test_cwt, 
    y_test=y_test, 
    epochs=50, 
    batch_size=64
)

print("\n--- CWT Deep Vision Results ---")
print(f"Accuracy: {acc * 100:.2f}%")
print(f"Macro F1-Score: {f1 * 100:.2f}%")


Training CWT Deep Vision Network on cuda...
Epoch 1/50 | Loss: 2.8892
Epoch 10/50 | Loss: 1.4904
Epoch 20/50 | Loss: 1.2929
Epoch 30/50 | Loss: 1.1828
Epoch 40/50 | Loss: 1.1040
Epoch 50/50 | Loss: 1.0275

--- CWT Deep Vision Results ---
Accuracy: 56.28%
Macro F1-Score: 55.20%
